In [4]:
import sys
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from nilearn.glm.first_level import compute_regressor

sys.path.insert(0, '/home/amin/Research/Representation/Movie/movie_watching')
from cifti_io import get_cortex_vertex_indices, save_cifti_map

# ============================================================================
# 1. CONFIGURATION & PATHS
# ============================================================================
DATA_BASE       = Path("/home/amin/Research/Representation/Movie/data")
RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/sitmulus_regressor_cifti")
TIMING_CSV  = DATA_BASE / "HCP Data" / "HCP_7T_Movie_Clip_Timing.csv"

# 59k group-average CIFTI (same as used by rsa/run_analysis.sh)
FMRI_CIFTI   = '/home/amin/Research/Representation/Movie/data/preprocessed/average_sub/raw/group_average_raw_cortex_59k.dtseries.nii'
RUN_TRS_FILE = '/home/amin/Research/Representation/Movie/data/preprocessed/average_sub/raw/group_average_raw_run_trs.npy'

TR_DUR = 1.0

# ============================================================================
# 2. LOAD & FORMAT TIMING DATA
# ============================================================================
print("Parsing stimulus timing...")
df = pd.read_csv(TIMING_CSV, header=1)
df.columns = ['movie_block', 'start_time', 'end_time', 'duration', 'comment']

stim_blocks = df[~df['movie_block'].astype(str).isin(['0', '0.0', 'nan', 'NaN'])].copy()
stim_blocks = stim_blocks.dropna(subset=['start_time', 'duration'])

onsets_abs = stim_blocks['start_time'].values
durations  = stim_blocks['duration'].values
print(f"Found {len(onsets_abs)} valid stimulus blocks.")

# ============================================================================
# 3. LOAD 59k fMRI DATA
# ============================================================================
print("Loading 59k group-average CIFTI...")
fmri_img  = nib.load(FMRI_CIFTI)
fmri_data = fmri_img.get_fdata(dtype=np.float32).T   # (59412, T)

RUN_TRS    = list(np.load(RUN_TRS_FILE).astype(int))
TOTAL_TRS  = fmri_data.shape[1]
RUN_OFFSETS = np.cumsum([0] + RUN_TRS[:-1])

# Determine left / right split from brain model axis
bm_axis = fmri_img.header.get_axis(1)
left_cifti_idx, right_cifti_idx = get_cortex_vertex_indices(bm_axis)
n_left  = len(left_cifti_idx)
n_right = len(right_cifti_idx)
print(f"CIFTI: {n_left} L + {n_right} R = {n_left+n_right} total grayordinates, {TOTAL_TRS} TRs")

fmri_left_raw  = fmri_data[:n_left, :]   # (n_left,  T)
fmri_right_raw = fmri_data[n_left:, :]   # (n_right, T)

valid_mask_left  = fmri_left_raw.std(axis=1)  > 0
valid_mask_right = fmri_right_raw.std(axis=1) > 0

# Group events by run to prevent HRF bleed-over at run boundaries
run_events = [[], [], [], []]
for onset, duration in zip(onsets_abs, durations):
    run_idx = 0
    for i, offset in enumerate(RUN_OFFSETS):
        if onset >= offset:
            run_idx = i
    rel_onset = onset - RUN_OFFSETS[run_idx]
    run_events[run_idx].append([rel_onset, duration, 1.0])

# ============================================================================
# 4. PER-RUN fMRI PROCESSING FUNCTION
# ============================================================================
def process_fmri_per_run(fmri_raw, valid_mask):
    processed_runs = []
    current_tr = 0
    for n_trs in RUN_TRS:
        run_data = fmri_raw[:, current_tr : current_tr + n_trs]
        run_proc = run_data - np.mean(run_data, axis=1, keepdims=True)
        run_std = np.ones((run_data.shape[0], 1))
        run_std[valid_mask, 0] = np.std(run_data[valid_mask, :], axis=1) + 1e-8
        run_proc = run_proc / run_std
        processed_runs.append(run_proc)
        current_tr += n_trs
    return np.hstack(processed_runs)

# ============================================================================
# 5. COMPUTE THE 4 VARIANTS
# ============================================================================
print("\n" + "="*50)
print("STARTING VARIANT GENERATION")
print("="*50)

for use_hrf in [False, True]:
    if use_hrf:
        print("\n--- Model: SPM HRF ---")
        hrf_lbl = "hrf"
        reg_runs = []
        for run_idx, n_trs in enumerate(RUN_TRS):
            if len(run_events[run_idx]) > 0:
                events_arr = np.array(run_events[run_idx])
                frame_times = np.arange(n_trs) * TR_DUR
                convolved_reg, _ = compute_regressor(events_arr.T, hrf_model='spm', frame_times=frame_times)
                reg_runs.append(convolved_reg.flatten())
            else:
                reg_runs.append(np.zeros(n_trs))
        base_regressor = np.concatenate(reg_runs)
    else:
        print("\n--- Model: 5s Fixed Delay Boxcar ---")
        hrf_lbl = "5sdelay"
        base_regressor = np.zeros(TOTAL_TRS)
        for onset, duration in zip(onsets_abs, durations):
            start_idx = min(max(int(np.round(onset + 5.0)), 0), TOTAL_TRS)
            end_idx   = min(max(int(np.round(onset + duration + 5.0)), 0), TOTAL_TRS)
            base_regressor[start_idx:end_idx] = 1.0

        print(f"  -> Processing: {hrf_lbl}")

        reg_mean = np.mean(base_regressor)
        base_regressor_c = (base_regressor - reg_mean) / (np.std(base_regressor) + 1e-8)
        left_proc = process_fmri_per_run(fmri_left_raw,valid_mask_left)
        right_proc = process_fmri_per_run(fmri_right_raw, valid_mask_right)

        def get_correlation(data_array, regressor):
            data_cent = data_array - np.mean(data_array, axis=1, keepdims=True)
            reg_cent  = regressor  - np.mean(regressor)
            cov       = np.dot(data_cent, reg_cent)
            ss_data   = np.sum(data_cent**2, axis=1)
            ss_reg    = np.sum(reg_cent**2)
            return cov / (np.sqrt(ss_data * ss_reg) + 1e-8)

        left_corr  = get_correlation(left_proc,  base_regressor)
        right_corr = get_correlation(right_proc, base_regressor)

        # Combine left + right → (59412,) — already in grayordinate space, no masking needed
        combined = np.concatenate([
            np.nan_to_num(left_corr,  nan=0.0),
            np.nan_to_num(right_corr, nan=0.0),
        ])

        RESULTS_BASE.mkdir(parents=True, exist_ok=True)
        map_output = str(RESULTS_BASE / f"HCP_movie_stimulus_correlation_{hrf_lbl}_{norm_lbl}.dscalar.nii")
        save_cifti_map(combined, FMRI_CIFTI, map_output,
                       map_name=f"Stimulus_Correlation_{hrf_lbl}_{norm_lbl}")
        print(f"  [SUCCESS] Saved CIFTI map: {Path(map_output).name}")

print("\nAll 4 maps generated successfully.")

Parsing stimulus timing...
Found 18 valid stimulus blocks.
Loading 59k group-average CIFTI...
CIFTI: 54216 L + 54225 R = 108441 total grayordinates, 3655 TRs

STARTING VARIANT GENERATION

--- Model: 5s Fixed Delay Boxcar ---
  -> Processing: 5sdelay
  [SUCCESS] Saved CIFTI map: HCP_movie_stimulus_correlation_5sdelay_normalized.dscalar.nii

--- Model: SPM HRF ---

All 4 maps generated successfully.
